In [139]:
import pandas as pd
import numpy as np 
df=pd.read_csv(r'Tweets.csv')
df.head(5)

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [140]:
x=df['text'].astype(str).tolist()
x=x[:1000]
y=df['airline_sentiment'].values.reshape(-1,1)

In [141]:
from sklearn.preprocessing import OneHotEncoder
en=OneHotEncoder(sparse_output=True)
one_hot=en.fit_transform(y).toarray()
one_hot=one_hot[:1000]

In [142]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
tokenizer=Tokenizer()
tokenizer.fit_on_texts(x)
seq=tokenizer.texts_to_sequences(x)
padded=pad_sequences(seq,padding='post')

In [143]:
len(tokenizer.word_index)

3248

In [144]:
padded.shape

(1000, 32)

In [145]:
padded[1]

array([   3,  404,  405,  547, 1297,    1,    4,  160, 1298,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0])

In [146]:
one_hot_x=to_categorical(padded-1,num_classes=len(tokenizer.word_index))


In [147]:
one_hot_x.shape

(1000, 32, 3248)

In [148]:
one_hot_x.shape

(1000, 32, 3248)

In [149]:
hidden_size=4
output_size=3
vacub_size=one_hot_x.shape[2]
learning_rate=0.01
epochs=1000
input_size=one_hot_x.shape[0]
timesteps=one_hot_x.shape[1]

In [153]:
class RNN:
    def __init__(self,hidden_size,output_size,vacub_size,learning_rate,epochs,input_size,timesteps):
        self.hidden_size=hidden_size
        self.output_size=output_size
        self.vacub_size=vacub_size
        self.learning_rate=learning_rate
        self.epochs=epochs
        self.timesteps=timesteps
        self.input_size=input_size
        self.wx=np.random.randn(self.hidden_size,self.vacub_size)*0.01
        self.wh=np.random.randn(self.hidden_size,self.hidden_size)*0.01
        self.wout=np.random.randn(self.hidden_size,self.output_size)*0.01
        self.hiddenstates=np.zeros((self.input_size,self.timesteps,self.hidden_size))
        self.bias=np.zeros((1,self.hidden_size))
        self.outbias=np.zeros((1,self.output_size))
        self.gradientswx=np.zeros_like(self.wx)
        self.biasgrads=np.zeros_like(self.bias)
        self.gradientswh=np.zeros_like(self.wh)
        self.loss_hist=[]
        self.acc_hist=[]
    def softmax(self,z):
        expz=np.exp(z)
        return expz/np.sum(expz,axis=1,keepdims=True)
    def forward_pass(self,x):
        for i in range(self.timesteps):
            self.hiddenstates[:,i,:]=self.relu((x[:,i,:]@self.wx.T)+self.hiddenstates[:,i-1,:]@self.wh.T+self.bias)
        z=self.hiddenstates[:,-1,:]@self.wout+self.outbias
        y_pred=self.softmax(z)
        return y_pred
    def relu(self,z):
        return np.maximum(0,z)
    def reluderivative(self,z):
        return np.where(z>0,1,0)
    def backward_pass(self,y,y_pred,x):
        douz=y_pred-y
        gradientswout=self.hiddenstates[:,-1,:].T@douz
        error_hidden_last=douz@self.wout.T
        error_out=error_hidden_last
        biasout=np.sum(douz,axis=0)
        for i in reversed(range(self.timesteps)):
            h_t=self.hiddenstates[:,i,:]
            #dela=error_out*(1-h_t**2)
            dela=error_out*self.reluderivative(h_t)
            self.gradientswx+=dela.T@x[:,i,:]
            self.biasgrads+=np.sum(dela,axis=0)
            if i>0:
                self.gradientswh+=dela.T@self.hiddenstates[:,i-1,:]
            error_out=dela@self.wh.T
        self.wx-=self.learning_rate*self.gradientswx
        self.wh-=self.gradientswh*self.learning_rate
        self.bias-=self.biasgrads*self.learning_rate
        self.outbias-=self.learning_rate*np.sum(douz,axis=0)
    def accuracy(self,y_pred,y):
        y_true=np.argmax(y,axis=1)
        y_pred_true=np.argmax(y_pred,axis=1)
        return np.mean(y_true==y_pred_true)
    def loss(self,y_pred,y):
        loss=-np.sum(y*np.log(y_pred))
        return loss
    def training(self,x,y):
        for epoch in range(self.epochs):
            y_pred=self.forward_pass(x)
            self.backward_pass(y,y_pred,x)
            loss=self.loss(y_pred,y)
            self.loss_hist.append(loss)
            accuracy=self.accuracy(y_pred,y)
            self.acc_hist.append(accuracy)
            if epoch%10==0:
                print(f'epoch {epoch} accuracy {self.acc_hist[epoch]} loss {self.loss_hist[epoch]}') 
            

In [154]:
rnn=RNN(hidden_size,output_size,vacub_size,learning_rate,epochs,input_size,timesteps)

In [155]:
rnn.training(one_hot_x,one_hot)

epoch 0 accuracy 0.215 loss 1098.6505281938694
epoch 10 accuracy 0.272 loss 3164.310192170015
epoch 20 accuracy 0.214 loss 1910.1527032231002
epoch 30 accuracy 0.272 loss 3049.491001043244
epoch 40 accuracy 0.214 loss 2028.9273201759033
epoch 50 accuracy 0.272 loss 3082.903644380591
epoch 60 accuracy 0.214 loss 1996.8203995517933
epoch 70 accuracy 0.272 loss 3074.6679293748743
epoch 80 accuracy 0.214 loss 2004.960614384339
epoch 90 accuracy 0.272 loss 3076.8179858325375
epoch 100 accuracy 0.214 loss 2002.8485262192498
epoch 110 accuracy 0.272 loss 3076.264098435364
epoch 120 accuracy 0.214 loss 2003.3935347021134
epoch 130 accuracy 0.272 loss 3076.407292878463
epoch 140 accuracy 0.214 loss 2003.2526953408237
epoch 150 accuracy 0.272 loss 3076.370306860891
epoch 160 accuracy 0.214 loss 2003.28907704976
epoch 170 accuracy 0.272 loss 3076.379862302506
epoch 180 accuracy 0.214 loss 2003.2796779989578
epoch 190 accuracy 0.272 loss 3076.377393776689
epoch 200 accuracy 0.214 loss 2003.2821061

In [152]:
rnn.training(one_hot_x,one_hot)

epoch 0 accuracy 0.272 loss 1098.6880439261936
epoch 10 accuracy 0.272 loss 3165.415670515246
epoch 20 accuracy 0.214 loss 1909.7227750227303
epoch 30 accuracy 0.272 loss 3051.5446812402474
epoch 40 accuracy 0.214 loss 2026.9428284748833
epoch 50 accuracy 0.272 loss 3082.4124098660495
epoch 60 accuracy 0.214 loss 1997.3095450582878
epoch 70 accuracy 0.272 loss 3074.7982818346522
epoch 80 accuracy 0.214 loss 2004.8328334282703
epoch 90 accuracy 0.272 loss 3076.78455534514
epoch 100 accuracy 0.214 loss 2002.8814384308132
epoch 110 accuracy 0.272 loss 3076.2727510035234
epoch 120 accuracy 0.214 loss 2003.3850256235737
epoch 130 accuracy 0.272 loss 3076.405058650641
epoch 140 accuracy 0.214 loss 2003.2548931428632
epoch 150 accuracy 0.272 loss 3076.370884124202
epoch 160 accuracy 0.214 loss 2003.2885092396998
epoch 170 accuracy 0.272 loss 3076.3797131770907
epoch 180 accuracy 0.214 loss 2003.2798246851783
epoch 190 accuracy 0.272 loss 3076.377432302159
epoch 200 accuracy 0.214 loss 2003.28